# Name That LLM: the tuned encoder, blended with the champion

**Goal:** train the tuned encoder (from the
[tuning notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-encoder-tune)) on full
5-fold CV, blend it with our champion `lm_blend` (TF-IDF `lr` + a logistic regression on
Qwen 3 4B hidden states), and test, honestly, whether the three-model ensemble beats the
champion.

Part of the learning-in-public MLOps project
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P5.

**How we keep it honest:**
- **Four fold seeds.** Seed **7** chooses the blend weights. Seeds **42, 43, 44** are fresh
  folds that only report.
- **The A/B gate:** a prompt-level bootstrap on seed 42, comparing the ensemble with
  `lm_blend`. We promote only if the whole 95% interval of the difference is above 0.
- **The seed-42 fold models are saved** for the offline submission. Its test prediction is
  their average, and those are the same models whose out-of-fold predictions chose the
  weights.

In [ ]:
# Pinned to an exact commit, so this run is reproducible even after main moves on.
%pip install -q "name-that-llm[track] @ git+https://github.com/Ryqn0/name-that-llm@f3064b38e91d931435ab8b8f85027568c6d8a18e"

In [ ]:
import glob
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

from name_that_llm.blend import best_blend_weights, blend
from name_that_llm.config import BlendConfig, EncoderConfig, EnsembleConfig, load_config
from name_that_llm.data import LABELS, load_train
from name_that_llm.encoder import run_cv_encoder
from name_that_llm.lm import select_layers
from name_that_llm.metrics import prompt_bootstrap_ci, score
from name_that_llm.train import run_cv, run_cv_features

print(
    "torch",
    torch.__version__,
    "| GPUs",
    [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
)
assert torch.cuda.is_available(), "this notebook needs a GPU (accelerator: T4)"

# Tracking is optional: without the Kaggle Secret, everything still runs.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    TRACK = True
except Exception as exc:
    TRACK = False
    print(f"W&B tracking is OFF ({type(exc).__name__}). Results are still computed.")

WORK = Path("/kaggle/working")
train = load_train()
y, groups = train["model"].to_numpy(), train["prompt_id"].to_numpy()
labels = np.asarray(LABELS)
ENC = load_config("encoder", model_cls=EncoderConfig)  # the tuned encoder (Task 5)
BLEND = load_config("lm_blend", model_cls=BlendConfig)  # the champion's LM head and weight
TEXT = load_config(BLEND.text_config)
print(ENC)

# The champion's LM features: Qwen 3 4B hidden states, extracted by the lm-features
# notebook. Row alignment (P3b Review Focus 1): the features are matched to labels by
# POSITION, so a mismatch would make every number here meaningless. Stop right away.
ids_files = glob.glob("/kaggle/input/**/train_ids.npy", recursive=True)
assert len(ids_files) == 1, f"expected one train_ids.npy, found {ids_files}"
assert (np.load(ids_files[0], allow_pickle=True) == train["id"].to_numpy()).all(), (
    "row order differs!"
)
feature_files = glob.glob(f"/kaggle/input/**/features-{BLEND.lm_variation}.npy", recursive=True)
assert len(feature_files) == 1, f"expected one features file, found {feature_files}"
X_LM = select_layers(np.load(feature_files[0]), BLEND.layer, BLEND.layer_window)
print("LM features:", X_LM.shape)

## 1. Three models' out-of-fold predictions, on four fold seeds

For each seed: the encoder (5 fine-tuning runs, one per fold, on the GPU), the TF-IDF text
model and the LM head (both on the CPU). Only the seed-42 encoder fold models are saved:
they're what the submission will use.

In [ ]:
OOF = {}
for seed in (7, 42, 43, 44):
    start = time.time()
    save_dir = WORK / "encoder" if seed == 42 else None
    enc = run_cv_encoder(ENC.model_copy(update={"seed": seed}), train, save_dir=save_dir)
    text = run_cv(TEXT.model_copy(update={"seed": seed}), train)
    lm = run_cv_features(X_LM, train, C=BLEND.C, seed=seed)
    OOF[seed] = {"text": text.oof, "lm": lm.oof, "encoder": enc.oof}
    np.save(WORK / f"oof-encoder-seed{seed}.npy", enc.oof.astype(np.float32))
    print(
        f"==> seed {seed}: encoder {enc.score:.4f} | lr {text.score:.4f} | LM head "
        f"{lm.score:.4f} | {(time.time() - start) / 60:.0f} min",
        flush=True,
    )

## 2. The ensemble's weights, chosen on seed 7

An exact search over every combination of three weights in steps of 0.05 that sums to 1.
On a tie, the newcomer (the encoder) gets the smaller weight.

In [ ]:
def predict(probas, weights):
    return labels[blend(probas, weights).argmax(axis=1)]


o7 = OOF[7]
WEIGHTS, ensemble7 = best_blend_weights([o7["text"], o7["lm"], o7["encoder"]], y)
lm_blend7 = score(y, predict([o7["text"], o7["lm"]], [1 - BLEND.weight, BLEND.weight]))
print(f"weights (text, LM head, encoder): {WEIGHTS}")
print(f"seed 7: ensemble {ensemble7:.4f} vs lm_blend {lm_blend7:.4f} (the sweep measured 0.4038)")

## 3. The honest report on fresh folds, and the A/B gate

In [ ]:
rows, kept = [], {}
for seed in (42, 43, 44):
    o = OOF[seed]
    champion = predict([o["text"], o["lm"]], [1 - BLEND.weight, BLEND.weight])
    ensemble = predict([o["text"], o["lm"], o["encoder"]], WEIGHTS)
    rows.append(
        {
            "seed": seed,
            "lr": score(y, predict([o["text"]], [1])),
            "lm_blend": score(y, champion),
            "encoder": score(y, predict([o["encoder"]], [1])),
            "ensemble": score(y, ensemble),
        }
    )
    kept[seed] = (champion, ensemble)
report = pd.DataFrame(rows).set_index("seed")
report.loc["mean"] = report.mean()
# print(), not just the display: the downloadable run log only keeps printed text.
print(report.round(4).to_string())

diff, lo, hi = prompt_bootstrap_ci(y, kept[42][0], kept[42][1], groups, n_boot=2000, seed=0)
PROMOTE = lo > 0
print(
    f"ensemble minus lm_blend (seed 42): {diff:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}] -> "
    + ("PROMOTE the ensemble" if PROMOTE else "keep lm_blend: the gain isn't clear")
)

## 4. Outputs

In [ ]:
cfg = EnsembleConfig(name="ensemble", weights=WEIGHTS)
with open(WORK / "ensemble.yaml", "w") as f:
    yaml.safe_dump(cfg.model_dump(mode="json"), f, sort_keys=False)
report.to_csv(WORK / "ensemble_report.csv")
print(yaml.safe_dump(cfg.model_dump(mode="json"), sort_keys=False))

# The seed-42 fold models, for the submission (attached as this notebook's output).
for fold in sorted((WORK / "encoder").iterdir()):
    size = sum(p.stat().st_size for p in fold.rglob("*") if p.is_file())
    print(f"{fold.name}: {size / 2**20:.0f} MB")

if TRACK:
    import wandb

    run = wandb.init(
        project="name-that-llm",
        job_type="encoder-final",
        name="encoder-final",
        config={"encoder": ENC.model_dump(), "weights": list(WEIGHTS)},
    )
    run.summary.update({f"cv_{k}_mean": report.loc["mean", k] for k in report.columns})
    run.summary.update({"diff": diff, "ci_low": lo, "ci_high": hi, "promote": bool(PROMOTE)})
    run.finish()

## 5. Findings

From run v1 (2026-10-26, 122 minutes on Kaggle's T4: 20 fine-tuning runs of about 5 minutes
each).

**1. The ensemble wins on every fresh seed.** The weights chosen on seed 7 are 0.70 for the
text model, 0.15 for the LM head and 0.15 for the encoder. On the three fresh fold seeds:

| seed | `lr` | `lm_blend` | encoder alone | **ensemble** |
|---|---|---|---|---|
| 42 | 0.321 | 0.405 | 0.350 | **0.422** |
| 43 | 0.330 | 0.400 | 0.354 | **0.425** |
| 44 | 0.325 | 0.402 | 0.356 | **0.423** |
| mean | 0.325 | 0.402 | 0.353 | **0.423** |

**2. The A/B gate promotes it.** On seed 42 the ensemble beats `lm_blend` by +0.017, with a
95% interval of [+0.004, +0.030]: the whole interval is above 0. It's saved as
`configs/ensemble.yaml`.

**3. A weaker model still helps.** The encoder alone (0.353) is well below `lm_blend`
(0.402), yet a 15% share lifts the blend by +0.021. That only works because its mistakes
differ from the other two models': it's fine-tuned end to end on the raw text, while they
are linear models on fixed features (word and character n-grams, frozen hidden states).

**4. Fine-tuning is noisy, so part of the tuning gain was luck.**
- On all five seed-7 folds, the tuned config scored 0.359, against 0.344 for the default in
  the encoder-select notebook: +0.015, half of the +0.030 the tuning measured on two folds.
- The same config, folds and seed scored 0.360 on folds 0 and 1 here, against 0.374 in the
  tuning run. The only difference is the run itself: GPU training isn't bit-for-bit
  reproducible (some CUDA operations add numbers in a varying order), and over 5 epochs the
  tiny differences grow. So a difference of about 0.015 on two folds can be pure noise.

**5. The weights weren't overfit to seed 7.** There the ensemble led `lm_blend` by +0.025;
on the fresh seeds it led by +0.017, +0.025 and +0.021.

**Next:** the offline submission (Task 7) averages the five seed-42 fold models (289 MB
each) on the test set, then blends them with the two CPU models using these weights. It
replaces the champion only if the public LB beats 0.389.